# Download ENSO Indicies Data

This script downloads ONI, RONI, and TNI historical data  
and compiles them into a data frame 
then converts  
then stores in the indicies_data/first_pass folder  

In [1]:
#dependencies
from pathlib import PosixPath
from pathlib import Path
import pandas as pd
import xarray as xr


In [2]:
#makes data directory structure
indicies_dir=Path("../indicies_data/first_pass")
indicies_dir.mkdir(parents=True, exist_ok=True)

# save name
output_roni_oni_tni_indicies= indicies_dir/'enso_historical_indicies.csv'

In [3]:
#get the data sources from NOAA and University of Miami into dataframes 

'''
direct web->notebook access encountered certification errors I could not resolve 
this data was manually placed into ../indicies_data/first_pass
the links are for source transparancy

oni_historic        ->     https://bmcnoldy.earth.miami.edu/tropics/oni/ONI_NINO34_v6.txt
tni_historic_csv    ->     https://psl.noaa.gov/data/timeseries/month/data/tni.long.csv
tni_historic_nc     ->     https://psl.noaa.gov/data/timeseries/month/data/tni.long.nc (this set may not be used in the workflow)
roni_historic       ->     https://bmcnoldy.earth.miami.edu/tropics/roni/RONI_NINO34_v6.txt
'''



#assign file paths
oni_historic=Path(indicies_dir/'ONI_NINO34_v6.txt')
roni_historic=Path(indicies_dir/'RONI_NINO34_v6.txt')
tni_historic=Path(indicies_dir/'tni.long.csv')

#read txt/csv files and init their variables

#oni
oni_historic_df=pd.read_csv(
    oni_historic,
    skiprows=10,
    sep=r"\s+",
    na_values=[-99.99],
)

#roni
roni_historic_df=pd.read_csv(
    roni_historic,
    skiprows=10,
    sep=r"\s+",
    na_values=[-99.99],
    )

#tni
tni_historic_df = pd.read_csv(
    tni_historic,
    header=0,
    names=["Date", "TNI"],      #replaces link in header with just date and tni
    parse_dates=["Date"],       #tells pandas these are dates 
    na_values=[-99.99,-9999],
)

#read nc file
#xr.open_dataset(tni_historic_nc)


In [4]:
#uncomment this to see the data dims and range

# display(tni_historic_df)
# display(oni_historic_df)
# display(roni_historic_df)

In [5]:
#create single date column for ONI and for RONI
#align date's of the data 

'''
ONI and RONI index have separate year and month columns
this TNI dataset also only goes back to 1870
'''

#form a single date column for oni 
oni_historic_df['Date']=pd.to_datetime(
    {
        'year':oni_historic_df['YEAR'],
        'month':oni_historic_df['MON/MMM'],
        'day':1
    }
)


#form a single date colum for roni 
roni_historic_df['Date']=pd.to_datetime(
    {
        'year':roni_historic_df['YEAR'],
        'month':roni_historic_df['MON'],
        'day':1
    }
)

In [6]:
# uncomment this to see how the dataframes were modified
# display(tni_historic_df)
#display(oni_historic_df)
# display(roni_historic_df)

In [7]:
#combine dataframes of oni and roni
#outer joins on the new column (key) DATE

oni_roni_historic_df=pd.merge(
    left=oni_historic_df[["Date","ONI"]], 
    right=roni_historic_df[["Date","RONI"]],
    on='Date', 
    how='outer'
)

#combine oni/roni df with tni

oni_roni_tni_historic_df=pd.merge(
    left=oni_roni_historic_df,
    right=tni_historic_df,
    on='Date',
    how='outer'
)



In [ ]:
''' 
Here is the final product
a dataframe of all three indicies
aligned by date
'''

oni_roni_tni_historic_df.to_csv(output_roni_oni_tni_indicies)
#uncomment to see
#display(oni_roni_tni_historic_df)
